# 01 · Data check

A quick visual sanity check of the processed tables in `data/processed/`. Nothing here changes any data.

To see the latest numbers, run `python -m sleeper_dash.pipeline` first, then **Run → Restart Kernel and Run All Cells**.

## Load the tables

Reads the four processed CSVs (IDs kept as text) and adds team names to the weekly table. The output shows how many rows each table has.

In [ ]:
import pandas as pd
import plotly.express as px
from sleeper_dash.config import load_config
from sleeper_dash.seasons import season_slice
from sleeper_dash.validate import load_tables

tables = season_slice(load_tables(), load_config().season)  # the saved tables hold every season; this notebook looks at the current one
tw = tables["team_weeks"].merge(tables["teams"][["roster_id", "team_name"]], on="roster_id")
tw = tw[~tw["is_playoff"]]  # regular season only
{name: len(table) for name, table in tables.items()}

## Current standings

Head-to-head record (**W, L, T**), wins against the weekly median (**med_W**), and **overall_W** = W + med_W, which is how Sleeper counts wins in this league. **PF** and **PA** are points for and against. Sorted by overall wins, then points for.

In [ ]:
count = lambda col, value: tw.groupby("team_name")[col].apply(lambda s: int((s == value).sum()))
standings = pd.DataFrame({
    "W": count("result", "W"), "L": count("result", "L"), "T": count("result", "T"),
    "med_W": count("median_result", "W"),
    "PF": tw.groupby("team_name")["points"].sum().round(2),
    "PA": tw.groupby("team_name")["opponent_points"].sum().round(2),
})
standings["overall_W"] = standings["W"] + standings["med_W"]
standings.sort_values(["overall_W", "PF"], ascending=False)

## Weekly scores by week

One box per week. The line inside each box is that week's **median**, the bar every team had to clear to win its median game. Each dot is one team; hover to see who.

In [ ]:
px.box(tw, x="week", y="points", points="all", hover_name="team_name",
       title="Team scores by week", labels={"points": "Points", "week": "Week"})

## Distribution of team-week scores

All team-weeks pooled into one histogram, followed by summary statistics. Look for a rough bell shape and any extreme scores worth double-checking.

In [ ]:
px.histogram(tw, x="points", nbins=20, title="All team-week scores",
             labels={"points": "Points"}).show()
tw["points"].describe().round(1)

## Team-by-week heatmap

Each cell is one team's score in one week; darker means more points. Teams are sorted by average score, highest at the top.

In [ ]:
grid = tw.pivot(index="team_name", columns="week", values="points")
grid = grid.loc[grid.mean(axis=1).sort_values(ascending=False).index]
px.imshow(grid, text_auto=".1f", aspect="auto", color_continuous_scale="Greens",
          title="Points by team and week", labels={"x": "Week", "y": "", "color": "Points"})